# gguf-serve — any GGUF model, one public OpenAI-compatible API

Run the two cells below. The first clones the repo, the second does everything else:
installs the CUDA build of llama.cpp, downloads and verifies the model, loads it across
your GPUs, and serves a chat UI plus an OpenAI-compatible API on one public URL.

**Before you start**, set the accelerator to **GPU T4 x2**
(Kaggle: *Settings -> Accelerator*). The default model, Qwen3.8-27B at Q5_K_XL, needs
about 23 GiB of VRAM — that is two T4s.

On free Colab you only get a single 16 GB T4 and the default will not fit. Use an L4 or
A100 runtime, or serve a smaller model by changing the launch command in cell 2 to:

```
!python launch.py --model-file Qwen3.8-27B-UD-Q3_K_XL.gguf
```

First run takes roughly 15 minutes, most of it downloading the model.

In [ ]:
# 1 — Get the code
#
# Change REPO_URL if you forked the repo.

REPO_URL = "https://github.com/kishorsharma/gguf-serve.git"

import os
import subprocess
import urllib.error
import urllib.request
from pathlib import Path

# Without this, an HTTPS URL that GitHub will not serve anonymously does not
# fail — it asks for a username and this cell hangs forever waiting for one.
# No username would help, so failing immediately is strictly better.
os.environ["GIT_TERMINAL_PROMPT"] = "0"

workdir = next(
    (p for p in (Path("/kaggle/working"), Path("/content")) if p.is_dir()),
    Path.cwd(),
)
repo = workdir / "gguf-serve"

_done = 0
_total = 2 if (repo / ".git").is_dir() else 3


def step(text):
    global _done
    _done += 1
    print(f"\n[{_done}/{_total}] {text}", flush=True)


def run(*cmd, cwd=None):
    print("   $", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd).returncode


def anonymous_clone_works(url):
    """Ask GitHub whether it will hand this repo to an anonymous client.

    Cheap pre-flight: it turns the credential prompt, which reads as a hang,
    into one line of explanation before we ever invoke git.
    """
    probe = url.removesuffix(".git") + ".git/info/refs?service=git-upload-pack"
    try:
        with urllib.request.urlopen(probe, timeout=30) as response:
            return response.status == 200, f"HTTP {response.status}"
    except urllib.error.HTTPError as error:
        return False, f"HTTP {error.code}"
    except Exception as error:  # DNS, TLS, no network
        return False, f"{type(error).__name__}: {error}"


if (repo / ".git").is_dir():
    step(f"{repo} already exists, updating it")
    if run("git", "pull", "--ff-only", "--progress", cwd=repo) == 0:
        print("   [ok] up to date")
    else:
        print("   [!] update failed — continuing with the copy already on disk")
else:
    step(f"Checking {REPO_URL}")
    reachable, detail = anonymous_clone_works(REPO_URL)
    if not reachable:
        print(f"   [x] GitHub will not serve that repo anonymously ({detail}).")
        print("       Usually one of:")
        print("       - it has not been pushed yet: create the repo on GitHub,")
        print("         then `git push -u origin main` from your machine")
        print("       - the owner or name in REPO_URL above is a typo")
        print("       - it is private, so an anonymous clone cannot see it. Use")
        print("         https://<token>@github.com/<owner>/<repo>.git instead")
        raise SystemExit("cannot clone REPO_URL")
    print(f"   [ok] reachable ({detail})")

    step(f"Cloning into {repo}")
    if run("git", "clone", "--depth", "1", "--progress", REPO_URL, str(repo)) != 0:
        raise SystemExit("git clone failed — see the output above")
    print("   [ok] cloned")

step("Entering the repo")
os.chdir(repo)
print("   [ok] working directory is", Path.cwd())

In [ ]:
# 2 — Install, download, load, serve
#
# Leave this cell running. The server lives inside it, so stopping the cell
# stops the server and kills the public URL.
#
# Watch for the https://....gradio.live line in the output — that is your
# public URL. Open it for the chat UI, add /docs for the API reference,
# or point any OpenAI client at <url>/v1.
#
# To serve a different model, add:
#   --model-repo <hf-repo> --model-file <file.gguf>

!python launch.py

## Using the API from anywhere

Once the public URL is up, any OpenAI client works against it. Run this from your
laptop, not from this notebook:

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://YOUR-ID.gradio.live/v1",
    api_key="not-used",  # this server does not check keys
)

response = client.chat.completions.create(
    model="qwen3.8-27b-ud-q5-k-xl",
    messages=[{"role": "user", "content": "Hello!"}],
)

print(response.choices[0].message.content)
```

`GET /health` reports the exact model id if you changed the model. The URL is
temporary: it is gone as soon as this notebook stops.

## Notes

- **The model is downloaded to `/tmp` and lost on restart.** `/kaggle/working` is
  capped at 20 GB, which is too small. To keep it, add the GGUF as a Kaggle dataset
  and pass `--model-dir /kaggle/input/<your-dataset>`.
- **Anyone with the public link can use the model.** There is no authentication.
  Use `--no-share` if you only want local access.
- To change the context size, GPU split, or sampling defaults, edit
  `ggufserve/config.py`. See `docs/configuration.md`.